# DriveSense-VLM — 05: Gradio Demo

**GPU**: A100 or T4 | **Time**: ~5 min | **Cost**: ~2 CU

Interactive Gradio demo: upload a dashcam image, get structured hazard detection JSON with bounding-box visualization.

> ⚠️ **Before running**: Runtime → Change runtime type → **T4 GPU** (or A100)
>
> **Prerequisites**: `02_optimization.ipynb` must have produced the quantized model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)

!ln -sfn {PROJECT_ROOT}/data {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs
sys.path.insert(0, f"{REPO_ROOT}/src")

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ Repo root    : {REPO_ROOT}")
print(f"✓ Outputs root : {OUTPUTS_ROOT}")

In [ ]:
!pip install gradio transformers peft accelerate "bitsandbytes>=0.46.1" "torchao>=0.16.0" Pillow -q 2>&1 | tail -3

import torch
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

assert os.path.exists(f"{OUTPUTS_ROOT}/quantized_model"), \
    "Missing quantized_model — run notebook 02_optimization.ipynb first"
print("✓ Quantized model found")

import drivesense
print(f"✓ drivesense package imported")

In [ ]:
import json, glob, html, time
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image, ImageDraw, ImageFont
import gradio as gr

# ── Config ────────────────────────────────────────────────────────────────────
QUANTIZED_MODEL_DIR = f"{OUTPUTS_ROOT}/quantized_model"
MERGED_MODEL_DIR    = f"{OUTPUTS_ROOT}/merged_model"
EXAMPLE_IMAGES = sorted(glob.glob(f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images/*.jpg"))[:6]
DEFAULT_MAX_TOKENS = 200

PROMPT = (
    "Analyze this dashcam image for safety hazards. Return JSON with hazards array "
    "containing bbox_2d (normalized 0-1000), label, severity (low/medium/high/critical), "
    "reasoning, and action for each hazard. Include scene_summary and ego_context "
    "(weather, time_of_day, road_type)."
)

# Modern UI design palette (hex) — boxes, cards and badges.
SEVERITY_HEX = {
    "critical": "#DC2626",
    "high":     "#EA580C",
    "medium":   "#CA8A04",
    "low":      "#16A34A",
    "no_hazard": "#2563EB",
}

def _hex_to_rgb(hex_color):
    h = hex_color.lstrip("#")
    return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16))

def _severity_hex(sev):
    return SEVERITY_HEX.get(str(sev).lower(), SEVERITY_HEX["no_hazard"])

def _format_latency(ms):
    return f"{ms/1000:.1f}s" if ms >= 1000 else f"{ms:.0f} ms"

# ── Load model (once) ─────────────────────────────────────────────────────────
print("Loading NF4 quantized model…")
_processor = AutoProcessor.from_pretrained(MERGED_MODEL_DIR)
_model = AutoModelForImageTextToText.from_pretrained(
    QUANTIZED_MODEL_DIR, device_map="auto", torch_dtype=torch.bfloat16,
)
_model.eval()
print(f"✓ Model loaded  |  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Helpers ───────────────────────────────────────────────────────────────────
def _parse_json(text):
    """Extract first JSON object; strip ```json fences."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1] if "\n" in text else text
        if text.endswith("```"):
            text = text[:-3].rstrip()
    start, end = text.find("{"), text.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return {"hazards": [], "scene_summary": text, "ego_context": {}}

def _get_font(size=16):
    for name in ("DejaVuSans.ttf", "Arial.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default()

def _text_size(draw, text, font):
    try:
        l, t, r, b = draw.textbbox((0, 0), text, font=font)
        return r - l, b - t
    except Exception:
        return len(text) * 8, 14

def _draw_boxes(image, ann):
    """Severity-coded boxes on a full-brightness copy (thick outline + label bg)."""
    base = image.convert("RGB").copy()
    draw = ImageDraw.Draw(base)
    font = _get_font(16)
    w, h = base.size
    for hazard in ann.get("hazards", []):
        bbox = hazard.get("bbox_2d", [])
        if len(bbox) != 4:
            continue
        sev   = str(hazard.get("severity", "no_hazard")).lower()
        label = str(hazard.get("label", "hazard"))
        rgb   = _hex_to_rgb(_severity_hex(sev))
        x1 = int(bbox[0] * w / 1000); y1 = int(bbox[1] * h / 1000)
        x2 = int(bbox[2] * w / 1000); y2 = int(bbox[3] * h / 1000)
        x1, x2 = sorted((x1, x2)); y1, y2 = sorted((y1, y2))
        draw.rectangle([x1, y1, x2, y2], outline=rgb, width=4)
        text = f"{label} · {sev}"
        tw, th = _text_size(draw, text, font)
        ty = y1 - th - 8 if y1 - th - 8 >= 0 else y1 + 2
        draw.rectangle([x1, ty, x1 + tw + 10, ty + th + 6], fill=rgb)
        draw.text((x1 + 5, ty + 3), text, fill=(255, 255, 255), font=font)
    return base

def _badge(text, bg="#475569"):
    return (f'<span style="display:inline-block;padding:2px 10px;margin:2px 4px 2px 0;'
            f'border-radius:9999px;background:{bg};color:#fff;font-size:12px;'
            f'font-weight:600;">{html.escape(text)}</span>')

def _summary_html(ann, ms):
    hazards = ann.get("hazards", [])
    n = len(hazards)
    count_color = "#16A34A" if n == 0 else "#DC2626"
    scene = ann.get("scene_summary", "") or "—"
    ego = ann.get("ego_context", {}) or {}
    ego_badges = "".join(
        _badge(f"{k.replace('_', ' ')}: {ego[k]}")
        for k in ("weather", "time_of_day", "road_type") if ego.get(k)
    ) or '<span style="color:#94a3b8;font-size:13px;">No scene context</span>'
    return f"""
<div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;padding:14px 16px;
            border:1px solid #e2e8f0;border-radius:12px;background:#f8fafc;">
  <div style="text-align:center;min-width:90px;">
    <div style="font-size:38px;font-weight:800;line-height:1;color:{count_color};">{n}</div>
    <div style="font-size:12px;color:#64748b;text-transform:uppercase;letter-spacing:.5px;">hazard{'s' if n != 1 else ''}</div>
  </div>
  <div style="border-left:1px solid #e2e8f0;padding-left:16px;flex:1;min-width:200px;">
    <div style="font-size:13px;color:#334155;margin-bottom:6px;">⏱ <b>{html.escape(_format_latency(ms))}</b> inference</div>
    <div style="font-size:13px;color:#475569;margin-bottom:8px;">{html.escape(scene)}</div>
    <div>{ego_badges}</div>
  </div>
</div>""".strip()

def _hazard_card(hazard):
    sev = str(hazard.get("severity", "no_hazard")).lower()
    color = _severity_hex(sev)
    label = str(hazard.get("label", "hazard"))
    reasoning = str(hazard.get("reasoning", "") or "—")
    action = str(hazard.get("action", "") or "—")
    return f"""
<div style="border:1px solid #e2e8f0;border-left:5px solid {color};border-radius:10px;
            padding:12px 14px;margin-bottom:10px;background:#fff;">
  <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:6px;">
    <span style="font-size:15px;font-weight:700;color:{color};">{html.escape(label)}</span>
    {_badge(sev.upper(), color)}
  </div>
  <div style="font-size:13px;color:#334155;margin-bottom:6px;"><b>Why:</b> {html.escape(reasoning)}</div>
  <div style="font-size:13px;color:#334155;"><b>Action:</b> {html.escape(action)}</div>
</div>""".strip()

def _hazards_html(ann):
    hazards = ann.get("hazards", [])
    if not hazards:
        return ('<div style="padding:18px;text-align:center;border:1px dashed #86efac;'
                'border-radius:12px;background:#f0fdf4;color:#15803d;font-weight:600;">'
                "✓ No hazards detected — clear scene</div>")
    return "".join(_hazard_card(h) for h in hazards)

def _info_html(msg):
    return ('<div style="padding:18px;text-align:center;border:1px dashed #cbd5e1;'
            'border-radius:12px;background:#f8fafc;color:#64748b;">' + html.escape(msg) + "</div>")

def analyze(image, max_tokens):
    if image is None:
        return None, _info_html("Upload a dashcam image to begin."), "", "{}"
    image = image.convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text":  PROMPT},
    ]}]
    text   = _processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _processor(text=[text], images=[image], return_tensors="pt").to("cuda")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    torch.cuda.synchronize()
    ms  = (time.perf_counter() - t0) * 1000
    raw = _processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ann = _parse_json(raw)
    annotated = _draw_boxes(image, ann)
    return annotated, _summary_html(ann, ms), _hazards_html(ann), json.dumps(ann, indent=2)

# ── Gradio UI ─────────────────────────────────────────────────────────────────
TITLE = "DriveSense-VLM: Autonomous Vehicle Hazard Detection"
DESCRIPTION = (
    "Upload a dashcam frame and DriveSense-VLM detects rare, safety-critical road hazards — "
    "drawing bounding boxes and explaining the risk and recommended ego action.\n\n"
    "> ⏱ Inference runs on T4 GPU (~20–40s per image). Model: **Qwen2.5-VL-3B**, NF4 quantized.\n\n"
    "**Severity:** 🔴 Critical &nbsp; 🟠 High &nbsp; 🟡 Medium &nbsp; 🟢 Low"
)

with gr.Blocks(title=TITLE, theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# {TITLE}")
    gr.Markdown(DESCRIPTION)
    with gr.Row():
        with gr.Column(scale=1):
            img_in  = gr.Image(label="Dashcam Frame", type="pil", image_mode="RGB")
            max_tok = gr.Slider(50, 500, value=DEFAULT_MAX_TOKENS, step=10, label="Max tokens")
            run_btn = gr.Button("Detect Hazards", variant="primary")
        with gr.Column(scale=1):
            img_out       = gr.Image(label="Annotated Detection", type="pil")
            summary_panel = gr.HTML(value=_info_html("Upload a dashcam image to begin."))
            hazards_panel = gr.HTML()
            with gr.Accordion("Raw JSON output", open=False):
                json_out = gr.Code(label="", language="json", lines=18)

    run_btn.click(fn=analyze, inputs=[img_in, max_tok],
                  outputs=[img_out, summary_panel, hazards_panel, json_out])

    if EXAMPLE_IMAGES:
        gr.Examples(examples=[[p] for p in EXAMPLE_IMAGES], inputs=[img_in],
                    label="Example Dashcam Frames")

demo.launch(share=True, debug=False)


In [7]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("✓ HF_TOKEN set")

✓ HF_TOKEN set


In [8]:
os.chdir(REPO_ROOT)
!git pull

# Copy example images first
import shutil, glob, os
os.makedirs(f"{REPO_ROOT}/huggingface_space/examples", exist_ok=True)
images = sorted(glob.glob(f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images/*.jpg"))[:6]
for img in images:
    shutil.copy(img, f"{REPO_ROOT}/huggingface_space/examples/")
print(f"✅ Copied {len(images)} example images")

# Upload to HuggingFace
!pip install huggingface_hub -q
from huggingface_hub import login
login()  # will prompt for your HF token

!python scripts/upload_to_hf.py \
    --model-dir {OUTPUTS_ROOT}/quantized_model \
    --processor-dir {OUTPUTS_ROOT}/merged_model \
    --examples-dir {REPO_ROOT}/huggingface_space/examples \
    --repo-id jayanth7111/DriveSense-VLM

Already up to date.
✅ Copied 6 example images
02:28:02  INFO      Upload plan for repo: jayanth7111/DriveSense-VLM
02:28:02  INFO        Model files     (9):
02:28:02  INFO          - model.safetensors
02:28:02  INFO          - config.json
02:28:02  INFO          - generation_config.json
02:28:02  INFO          - processor_config.json
02:28:02  INFO          - quality_comparison.json
02:28:02  INFO          - quant_config.json
02:28:02  INFO          - tokenizer.json
02:28:02  INFO          - tokenizer_config.json
02:28:02  INFO          - chat_template.jinja
02:28:02  INFO        Processor files (7):
02:28:02  INFO          - model.safetensors
02:28:02  INFO          - config.json
02:28:02  INFO          - generation_config.json
02:28:02  INFO          - processor_config.json
02:28:02  INFO          - tokenizer.json
02:28:02  INFO          - tokenizer_config.json
02:28:02  INFO          - chat_template.jinja
02:28:02  INFO        Examples        (6):
02:28:02  INFO          - examples

In [9]:
# This pushes again — HF creates new commits for changed files
!python scripts/upload_to_hf.py \
    --model-dir {OUTPUTS_ROOT}/quantized_model \
    --processor-dir {OUTPUTS_ROOT}/merged_model \
    --examples-dir {REPO_ROOT}/huggingface_space/examples \
    --repo-id jayanth7111/DriveSense-VLM

02:30:59  INFO      Upload plan for repo: jayanth7111/DriveSense-VLM
02:30:59  INFO        Model files     (9):
02:30:59  INFO          - model.safetensors
02:30:59  INFO          - config.json
02:30:59  INFO          - generation_config.json
02:30:59  INFO          - processor_config.json
02:30:59  INFO          - quality_comparison.json
02:30:59  INFO          - quant_config.json
02:30:59  INFO          - tokenizer.json
02:30:59  INFO          - tokenizer_config.json
02:30:59  INFO          - chat_template.jinja
02:30:59  INFO        Processor files (7):
02:30:59  INFO          - model.safetensors
02:30:59  INFO          - config.json
02:30:59  INFO          - generation_config.json
02:30:59  INFO          - processor_config.json
02:30:59  INFO          - tokenizer.json
02:30:59  INFO          - tokenizer_config.json
02:30:59  INFO          - chat_template.jinja
02:30:59  INFO        Examples        (6):
02:30:59  INFO          - examples/n008-2018-08-01-15-16-36-0400__CAM_FRONT__153

In [ ]:
import glob, os
from IPython.display import Image as IPImage, display

# Get test images sorted
images = sorted(glob.glob(f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images/*.jpg"))
print(f"Total available: {len(images)} images\n")

# Display first 10 to visually pick a good one
for i, img_path in enumerate(images[:10]):
    print(f"\n[{i}] {os.path.basename(img_path)}")
    display(IPImage(filename=img_path, width=400))

In [ ]:
# Pick the index you liked
INDEX = 2  # change this
selected = images[INDEX]
print(f"Use this in Gradio: {selected}")

# Or download it to upload via the public Gradio link
from google.colab import files
files.download(selected)